# Loan Application Evaluator
## Multi-Agent Evaluation System for Financial Sector

This notebook demonstrates a sophisticated loan evaluation system using Strands agents with AWS Bedrock.

### Agents Involved:
1. **Credit Risk Analyst** - Evaluates creditworthiness and credit metrics
2. **Compliance Officer** - Ensures regulatory compliance and documentation
3. **Fraud Detection Specialist** - Identifies fraud patterns and inconsistencies
4. **Loan Officer** - Provides final decision and recommended terms

## Setup and Imports

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

# Add project to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from loan_evaluator import LoanEvaluator
from models import LoanApplication, EvaluationResult
from dotenv import load_dotenv

load_dotenv()

print("✓ Imports successful")
print(f"Project root: {project_root}")

✓ Imports successful
Project root: /Users/dias/Documents/GitHub/strands-agents-loan-evaluator/loan_evaluator


## Initialize Evaluator

In [2]:
# Initialize the loan evaluator with AWS Bedrock
evaluator = LoanEvaluator(
    model_name="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region="us-east-1",
    use_langsmith=True,  # Requires LANGSMITH_API_KEY and enables LangSmith traces
    include_reasoning_in_traces=True,  # Sends strengths/weaknesses/risks to LangSmith
)

print("✓ Evaluator initialized successfully")
print(f"Model: Claude 4.5 v1 Sonnet via Bedrock")
print(f"Agents: {list(evaluator.evaluators.keys())}")

✓ Evaluator initialized successfully
Model: Claude 4.5 v1 Sonnet via Bedrock
Agents: ['credit_analyst', 'compliance_officer', 'fraud_detector', 'loan_officer']


## Load Sample Loan Application

In [3]:
# Load sample application 1 (Strong applicant)
sample_data_path = Path("sample_data/loan_application_1.json")

with open(sample_data_path, "r") as f:
    app_data = json.load(f)

application = LoanApplication(**app_data)

# Display application summary
print(f"Applicant: {application.applicant_name}")
print(f"Requested Amount: ${application.requested_amount:,.2f}")
print(f"Loan Purpose: {application.loan_purpose}")
print(f"Credit Score: {application.credit_score}")
print(f"Employment: {application.employment_status} at {application.current_employer}")
print(f"Annual Income: ${application.annual_income:,.2f}")

Applicant: Sarah Johnson
Requested Amount: $350,000.00
Loan Purpose: Home Purchase
Credit Score: 755
Employment: Employed at Tech Solutions Inc.
Annual Income: $125,000.00


## Run Multi-Agent Evaluation

This will execute all specialized agents and compile their evaluations.

In [4]:
# Run the evaluation
print("Starting multi-agent evaluation...\n")
result = await evaluator.evaluate_async(application)

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)

Starting multi-agent evaluation...

Running Credit Risk Analysis...
Running Compliance Review...
Running Fraud Detection...
Getting Final Loan Officer Review...

EVALUATION COMPLETE


## Display Results

In [5]:
# Summary Results
print(f"\nAPPLICAN: {result.applicant_name}")
print(f"Evaluation Date: {result.evaluation_date}")
print(f"\nFINAL RECOMMENDATION: {result.final_recommendation.upper()}")
print(f"Overall Score: {result.overall_score:.1f}/100")
print(f"Average Confidence: {result.avg_confidence:.1%}")
print(f"\nRationale: {result.decision_rationale}")


APPLICAN: Sarah Johnson
Evaluation Date: 2026-04-26 11:16:08.011618

FINAL RECOMMENDATION: DENY
Overall Score: 36.0/100
Average Confidence: 95.8%

Rationale: After careful synthesis of all specialist reviews, I must recommend DENIAL of this loan application. While Ms. Johnson presents several positive attributes including an excellent 755 credit score, stable 8-year employment history, and complete documentation, there are fundamental mathematical impossibilities that prevent approval. The reported debt-to-income ratio of 144% means the applicant's existing debt obligations ($15,000/month) already exceed her total gross monthly income ($11,667/month) by 44%. This makes it mathematically impossible to service the requested mortgage payment while meeting existing obligations. Additionally, liquid assets of $85,000 are insufficient to cover the stated $100,000 down payment, creating a $15,000 funding gap. All three specialists (Credit Analyst, Compliance Officer, and Fraud Detective) ide

## Individual Agent Reviews

In [12]:
# Detailed agent reviews
for i, review in enumerate(result.reviews, 1):
    print(f"\n{'='*50}")
    print(f"{i}. {review.reviewer_name.upper()}")
    print(f"{'='*50}")
    print(f"Score: {review.score}/100")
    print(f"Recommendation: {review.recommended_action}")
    print(f"Confidence: {review.confidence:.1%}")
    
    print(f"\nStrengths:")
    for strength in review.strengths[:3]:
        print(f"  ✓ {strength}")
    
    print(f"\nWeaknesses:")
    for weakness in review.weaknesses[:3]:
        print(f"  ✗ {weakness}")
    
    print(f"\nRisks:")
    for risk in review.risks[:3]:
        print(f"  ⚠ {risk}")


1. CREDIT_RISK_ANALYST
Score: 35/100
Recommendation: deny
Confidence: 95.0%

Strengths:
  ✓ Excellent credit score of 755 indicates strong creditworthiness and responsible credit management
  ✓ Stable employment history with 8 years at current employer demonstrates job security
  ✓ Strong annual income of $125,000 plus $15,000 secondary income totaling $140,000

Weaknesses:
  ✗ CRITICAL: Debt-to-Income ratio of 144% is extremely high and well above acceptable lending standards (typically max 43-50%)
  ✗ Existing monthly debt obligations of $15,000 are unsustainably high relative to gross monthly income of ~$11,667
  ✗ Relatively short residence history of 3 years may indicate some instability

Risks:
  ⚠ SEVERE RISK: DTI of 144% indicates applicant is already over-leveraged and cannot support additional mortgage debt
  ⚠ High probability of payment default if income disruption occurs due to minimal debt service capacity
  ⚠ Existing $15,000 monthly debt obligations suggest potential u

## Recommended Terms (if approved)

In [13]:
if result.recommended_interest_rate or result.recommended_loan_amount:
    print("RECOMMENDED LOAN TERMS")
    print("="*50)
    if result.recommended_interest_rate:
        print(f"Interest Rate: {result.recommended_interest_rate:.2%}")
    if result.recommended_loan_amount:
        print(f"Loan Amount: ${result.recommended_loan_amount:,.2f}")
    if result.special_conditions:
        print(f"\nSpecial Conditions:")
        for condition in result.special_conditions:
            print(f"  • {condition}")
else:
    print("No specific terms recommended at this time.")

No specific terms recommended at this time.


## Export Results

In [14]:
# Export the evaluation to JSON
export_path = evaluator.export_result(result)
print(f"Results saved to: {export_path}")

Evaluation exported to: evaluation_Sarah Johnson_20260426_110757.json
Results saved to: evaluation_Sarah Johnson_20260426_110757.json


## Test with Additional Applications

In [15]:
# Load and evaluate additional sample applications
sample_files = list(Path("sample_data").glob("loan_application_*.json"))

print(f"Found {len(sample_files)} sample applications")
print("\nAvailable samples:")
for i, file in enumerate(sorted(sample_files), 1):
    with open(file) as f:
        app_data = json.load(f)
    print(f"{i}. {app_data['applicant_name']} - {app_data['loan_purpose']} (${app_data['requested_amount']:,.0f})")

Found 3 sample applications

Available samples:
1. Sarah Johnson - Home Purchase ($350,000)
2. Michael Chen - Business ($500,000)
3. James Rodriguez - Home Purchase ($275,000)


In [16]:
# Evaluate another application (e.g., sample 2)
sample_path = Path("sample_data/loan_application_2.json")

with open(sample_path) as f:
    app2_data = json.load(f)

app2 = LoanApplication(**app2_data)

print(f"Evaluating: {app2.applicant_name}")
print(f"Purpose: {app2.loan_purpose}")
print(f"Amount: ${app2.requested_amount:,.2f}\n")

result2 = await evaluator.evaluate_async(app2)
print(f"\nResult: {result2.final_recommendation.upper()} (Score: {result2.overall_score:.1f}/100)")

Evaluating: Michael Chen
Purpose: Business
Amount: $500,000.00

Running Credit Risk Analysis...
Running Compliance Review...
Running Fraud Detection...
Getting Final Loan Officer Review...
recommended_action
  Field required [type=missing, input_value={'reviewer_name': 'loan_o...n.", 'confidence': 0.99}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

Result: REQUEST_MORE_INFO (Score: 22.8/100)


## Performance Metrics

In [17]:
# Display evaluation statistics
print("EVALUATION STATISTICS")
print("="*50)
print(f"Number of reviews: {len(result.reviews)}")
print(f"Agents involved: {', '.join(result.agents_involved)}")
print(f"Average confidence: {result.avg_confidence:.1%}")
print(f"Final score distribution:")
print(f"  - Highest score: {max(r.score for r in result.reviews)}/100")
print(f"  - Lowest score: {min(r.score for r in result.reviews)}/100")
print(f"  - Average score: {sum(r.score for r in result.reviews) / len(result.reviews):.1f}/100")

EVALUATION STATISTICS
Number of reviews: 4
Agents involved: credit_risk_analyst, compliance_officer, fraud_detection_specialist, loan_officer
Average confidence: 73.8%
Final score distribution:
  - Highest score: 50/100
  - Lowest score: 25/100
  - Average score: 38.8/100
